In [22]:
# ============================================================
# 02_preprocessing.ipynb — NLP Preprocessing Pipeline
# ============================================================

In [23]:
import pandas as pd
import re
import nltk
import csv
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [24]:
nltk.download('stopwords')
nltk.download('wordnet')

df = pd.read_csv("../data/raw/Hotel_Reviews.csv")
print("Original shape:", df.shape)

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = re.sub(r'<.*?>', '', str(text))
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.lower().strip()
    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words]
    tokens = [lemmatizer.lemmatize(w) for w in tokens]
    return " ".join(tokens)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\windows\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\windows\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Original shape: (515738, 17)


In [25]:
# ── NEGATIVE REVIEWS ────────────────────────────────────────
neg = df[~df['Negative_Review'].str.strip().isin(['No Negative', ''])][
    ['Negative_Review', 'Hotel_Name', 'Average_Score', 'Reviewer_Score']
].copy()
neg.rename(columns={'Negative_Review': 'review_text'}, inplace=True)
neg['label'] = 'Negative'

In [26]:
# ── POSITIVE REVIEWS ────────────────────────────────────────
pos = df[~df['Positive_Review'].str.strip().isin(['No Positive', ''])][
    ['Positive_Review', 'Hotel_Name', 'Average_Score', 'Reviewer_Score']
].copy()
pos.rename(columns={'Positive_Review': 'review_text'}, inplace=True)
pos['label'] = 'Positive'

In [27]:
# ── COMBINE & INTERLEAVE (neg then pos per original row) ────
neg['_idx'] = neg.index
pos['_idx'] = pos.index

combined = pd.concat([neg, pos], ignore_index=True)
combined = combined.sort_values('_idx').drop(columns='_idx').reset_index(drop=True)

print("After unpivot:", combined.shape)
print(combined['label'].value_counts())

After unpivot: (866608, 5)
label
Positive    479609
Negative    386999
Name: count, dtype: int64


In [28]:
# ── PREPROCESS ──────────────────────────────────────────────
combined['clean_text'] = combined['review_text'].apply(preprocess)

In [29]:
# ── REMOVE EMPTY ────────────────────────────────────────────
combined = combined[combined['clean_text'].str.strip() != '']
combined = combined.dropna(subset=['clean_text'])

In [30]:
# ── FINAL COLUMNS ───────────────────────────────────────────
df_final = combined[['clean_text', 'label', 'Hotel_Name',
                      'Average_Score', 'Reviewer_Score']].copy()

print("\nFinal shape:", df_final.shape)
print(df_final['label'].value_counts())
print("\nSample (first 6 rows):")
print(df_final[['clean_text', 'label', 'Reviewer_Score']].head(6))


Final shape: (865601, 5)
label
Positive    479095
Negative    386506
Name: count, dtype: int64

Sample (first 6 rows):
                                          clean_text     label  Reviewer_Score
0  angry made post available via possible site us...  Negative             2.9
1                       park outside hotel beautiful  Positive             2.9
2  real complaint hotel great great location surr...  Positive             7.5
3  location good staff ok cute hotel breakfast ra...  Positive             7.1
4  room nice elderly bit difficult room two story...  Negative             7.1
5  great location nice surroundings bar restauran...  Positive             3.8


In [ ]:
# ── EXPORT ──────────────────────────────────────────────────
df_final.to_csv(
    '../data/processed/clean_reviews.csv',
    index=False,
    sep=',',
    encoding='utf-8',
    quoting=csv.QUOTE_ALL
)
print("\n✅ Saved!")

PermissionError: [Errno 13] Permission denied: '../data/processed/clean_reviews.csv'

: 